In [2]:
import urllib.parse, json, base64, hashlib, zlib, csv
import pandas as pd
import re, os

In [ ]:
def set_to_dict(lst):
    
    converted = {}
    
    for name, value in lst:
        
        converted[name] = value
        
    return converted

def set_to_invdict(lst):
    
    converted = {}
    
    for name, value in lst:
        
        converted[value] = name
        
    return converted

# Hashes & Encodings

In [ ]:
def url_encode(value):
    return urllib.parse.quote(value, safe='')

def base64_encode(value):
    return base64.b64encode(value.encode()).decode()

def md5_hash(value):
    return hashlib.md5(value.encode()).hexdigest()

def sha1_hash(value):
    return hashlib.sha1(value.encode()).hexdigest()

def sha256_hash(value):
    return hashlib.sha256(value.encode()).hexdigest()

# Values

In [ ]:
HASH_VALUES = ['useremail']

In [ ]:
def fetch_values(text):
    
    values = set()
    
    for line in text.splitlines():
        
        name, value = line.split(':')
        values.add((name, value))
    
    return values

def transform_values(values):
    
    transformed = set()
    
    for name, value in values:
        
        if name in HASH_VALUES:
            
            transformed.add((name, md5_hash(value)))
            transformed.add((name, sha1_hash(value)))
            transformed.add((name, sha256_hash(value)))
            
        transformed.add((name, url_encode(value)))
        transformed.add((name, base64_encode(value)))
    
    return transformed

# Domains

In [ ]:
DOMAIN_COLUMN = 'third_party'
DOMAIN_NAME_COLUMN = 'third_party_name'

FIRST_PARTY_NAMES = ["First-party", "First-Party", 'Grok']

In [ ]:
def fetch_domains(df):
    
    domains = set()
    
    for _, row in df.iterrows():
        
        domain = row[DOMAIN_COLUMN]
        name = row[DOMAIN_NAME_COLUMN]
        
        if name not in FIRST_PARTY_NAMES:
            
            domains.add((name, domain))
    
    return domains

# Hunter

In [ ]:
ACCOUNT = { "1": "No Auth", "2": "Free", "3": "Premium"}
CHAT = { "1": "Normal", "2": "Incognito"}
PRIVACY = { "1": "Default", "2": "Minimum", "3": "Maximum"}
CONSENT = { "0": "No consent banner", "1": "Ignore Consent", "2": "Reject All", "3": "Accept All"}
INTERACTION = { "1": "Normal", "2": "Share", "3": "Load"}

VALUES_FILE = ''
COLLECTED_FILE = ''

In [ ]:
CONVARTF_IDS_NAMES = ['conversationid', 'shareid']

In [ ]:
def regex_value(name, value):
    
    if (name in CONVARTF_IDS_NAMES):
        
        return rf'(?<!/)(?<!%2F){re.escape(value)}(?!/|%2F)'
    
    return rf'{re.escape(value)}'

def hunt_values(text, values):
    
    found_text = text
    found_values = []
    
    for name, value in values:
        
        regex = re.search(regex_value(name, value), text)

        if regex:
            
            found_values.append((name, value))
            found_text = re.sub(regex_value(name, value), f'<{name} -> {value}>', found_text)
    
    return found_text, found_values

In [ ]:

def hunt_header(header, values):
    
    if header['name'] not in ['cookie', 'set-cookie', ':path']:
        
        found_text, matches = hunt_values(header['value'], values)
        
        return found_text, [('headers', name, value) for name, value in matches]
    
    return header['value'], []

def hunt_cookie(cookie, values):
    
    found_text, matches = hunt_values(cookie['value'], values)
    
    return found_text, [('cookies', name, value) for name, value in matches]

def hunt_query(param, values):
    
    found_text, matches = hunt_values(param['value'], values)
    
    return found_text, [('query', name, value) for name, value in matches]

def hunt_postdata(postdata, values):
    
    text = postdata['text']
    
    try:
        
        raw = text.encode('latin-1')
        text = zlib.decompress(raw)
        
    except:
        pass
    
    found_text, matches = hunt_values(text, values)
    
    return found_text, [('postdata', name, value) for name, value in matches]

def hunt_request(request, values):
    
    matches = []
    
    domain = urllib.parse.urlparse(request['url']).netloc
    path = urllib.parse.urlparse(request['url']).path
    
    for header in request['headers']:
        
        found_text, found_matches = hunt_header(header, values)
        
        header['value'] = found_text
        matches.extend([(domain, path, *match) for match in found_matches])
        
    for cookie in request['cookies']:
        
        found_text, found_matches = hunt_cookie(cookie, values)
        
        cookie['value'] = found_text
        matches.extend([(domain, path, *match) for match in found_matches])
        
    for param in request['queryString']:
        
        found_text, found_matches = hunt_query(param, values)
        
        param['value'] = found_text
        matches.extend([(domain, path, *match) for match in found_matches])
        
    if 'postData' in request:
        
        found_text, found_matches = hunt_postdata(request['postData'], values)
        
        request['postData']['text'] = found_text
        matches.extend([(domain, path, *match) for match in found_matches])
        
    return matches

In [ ]:
def hunt_har(har, domains, values):
    
    matches = set()
    
    for entry in har['log']['entries']:
        
        request = entry['request']
        domain = urllib.parse.urlparse(request['url']).netloc
        
        if domain in domains.keys():
            
            matches.update([(domains[domain], *match) for match in hunt_request(request, values)])
            
    return matches

def hunt_hars(root, domains):
    
    matches = set()
    
    for path, _, files in os.walk(root):
        
        for file in files:
            
            if file.endswith('.har'):
                
                values = set()
                
                regex = re.search(r'(\w+)-A(\d)-P(\d)-T(\d)-C(\d)-S\d-\d{8}/I(\d)', path)
                llm = regex.group(1)
                account = regex.group(2)
                chat = regex.group(3)
                privacy = regex.group(4)
                consent = regex.group(5)
                interaction = regex.group(6)
                
                path_values = os.path.join(path, VALUES_FILE)
                path_collec = os.path.join(path, COLLECTED_FILE)
                path_har = os.path.join(path, file)
                
                with open(path_values, 'r') as f:
                    
                    text = f.read()
                    values |= fetch_values(text)
                    
                with open(path_collec, 'r') as f:
                    
                    text = f.read()
                    values |= fetch_values(text)
                    
                values |= transform_values(values)
                
                with open(path_har, 'r') as f:
                    
                    har = json.load(f)
                    found_matches = hunt_har(har, domains, values)
                    
                    matches |= set([(llm, *match, ACCOUNT[account], CHAT[chat], PRIVACY[privacy], CONSENT[consent], INTERACTION[interaction]) for match in found_matches])

    return matches

# Execution

In [ ]:
ROOT_DIR = ""           # Fill this parameter with the root location of all HAR files
DOMAINS_PATH = ""       # Fill this parameter with the location of the third party csv

df = pd.read_csv(DOMAINS_PATH)
domains_set = fetch_domains(df)
domains_invdict = set_to_invdict(domains_set)

matches = hunt_hars(ROOT_DIR, domains_invdict)

In [ ]:
for match in matches:
    print(match)

In [ ]:
OUTPUT_PATH = ""        # Fill this parameter with desired output location for the CSV

with open(OUTPUT_PATH, 'w', newline="") as f:
    
    writer = csv.writer(f)
    writer.writerows(matches)